In [11]:
K = 20
f2 = flows.rename(columns={"Source IP": "src_ip", "Destination IP": "dst_ip",
                           "Destination Port": "dst_port",
                           "Total Fwd Packets": "fwd", "Total Backward Packets": "bwd"})
f2["dst_port"] = pd.to_numeric(f2["dst_port"], errors="coerce")
f2 = f2.dropna(subset=["dst_port"]).copy()
f2["no_reply"] = (f2["bwd"] == 0).astype(float)
f2["chunk"] = f2.groupby(["file", "src_ip"], sort=False).cumcount() // K

def chunk_features(df, keys):
    g = df.groupby(keys, sort=False)
    w = g.agg(n=("dst_port", "size"), n_dst_ports=("dst_port", "nunique"),
              n_dst_hosts=("dst_ip", "nunique"), no_reply_share=("no_reply", "mean"),
              mean_fwd_pkts=("fwd", "mean"), mean_bwd_pkts=("bwd", "mean"))
    top = df.groupby(keys + ["dst_port"], sort=False).size().groupby(level=keys, sort=False).max()
    w["top_port_share"] = top / w["n"]
    return w[w["n"] == K]

w = chunk_features(f2, ["file", "src_ip", "chunk"])
fam = f2.groupby(["file", "src_ip", "chunk"], sort=False)["family"].agg(lambda s: s.mode().iat[0])
w = w.join(fam).reset_index()

grp = w.groupby(["file", "src_ip"], sort=False)["chunk"]
w["is_test"] = (grp.rank(method="first") / grp.transform("size")) > 0.7
print(w.shape)
w.groupby(["family", "is_test"]).size().unstack(fill_value=0)

(132494, 12)


is_test,False,True
family,,
benign,71371,33228
bruteforce,490,211
dos,13404,5735
portscan,5562,2384
web,77,32


In [12]:
FEATS = ["n_dst_ports", "n_dst_hosts", "top_port_share",
         "no_reply_share", "mean_fwd_pkts", "mean_bwd_pkts"]
tr, te = w[~w["is_test"]], w[w["is_test"]]

rfw = RandomForestClassifier(n_estimators=200, min_samples_leaf=2,
                             class_weight="balanced", n_jobs=-1, random_state=42)
rfw.fit(tr[FEATS], tr["family"])
pred = rfw.predict(te[FEATS])
print(classification_report(te["family"], pred, digits=4, zero_division=0))
labels = sorted(w["family"].unique())
print(pd.DataFrame(confusion_matrix(te["family"], pred, labels=labels), index=labels, columns=labels))
print(pd.Series(rfw.feature_importances_, index=FEATS).sort_values(ascending=False).round(4))
joblib.dump({"model": rfw, "features": FEATS}, "../models/rf_window_v1.joblib")

              precision    recall  f1-score   support

      benign     0.9797    0.9918    0.9857     33228
  bruteforce     0.9688    0.5877    0.7316       211
         dos     0.9950    0.9013    0.9458      5735
    portscan     0.9026    0.9878    0.9433      2384
         web     0.9500    0.5938    0.7308        32

    accuracy                         0.9767     41590
   macro avg     0.9592    0.8125    0.8674     41590
weighted avg     0.9773    0.9767    0.9763     41590

            benign  bruteforce   dos  portscan  web
benign       32955           4    15       254    0
bruteforce      87         124     0         0    0
dos            565           0  5169         0    1
portscan        29           0     0      2355    0
web              2           0    11         0   19
mean_fwd_pkts     0.3057
mean_bwd_pkts     0.2644
n_dst_ports       0.1535
top_port_share    0.1274
n_dst_hosts       0.0966
no_reply_share    0.0524
dtype: float64


['../models/rf_window_v1.joblib']

In [13]:
def zeek_windows(conn):
    d = conn.copy()
    d["ts"] = pd.to_numeric(d["ts"]); d = d.sort_values("ts")
    for c in ["orig_pkts", "resp_pkts"]:
        if c not in d.columns: d[c] = np.nan
        d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0)
    d["dst_port"] = pd.to_numeric(d["id.resp_p"], errors="coerce")
    d = d.rename(columns={"id.orig_h": "src_ip", "id.resp_h": "dst_ip",
                          "orig_pkts": "fwd", "resp_pkts": "bwd"})
    d["no_reply"] = (d["bwd"] == 0).astype(float)
    d = d.dropna(subset=["dst_port"]).copy()
    d["chunk"] = d.groupby("src_ip", sort=False).cumcount() // K
    return chunk_features(d, ["src_ip", "chunk"])

LOGS = Path("../data/captures")
for f in sorted(LOGS.rglob("conn.log")):
    conn = pd.read_json(f, lines=True)
    if len(conn) < 50:
        print("skipping tiny capture:", f.parent.name, len(conn)); continue
    ww = zeek_windows(conn)
    p = rfw.predict(ww[FEATS])
    print(f"\n{f.parent.name}: {len(conn)} flows -> {len(ww)} bundles")
    print(pd.Series(p).value_counts().to_dict())

skipping tiny capture: zeek_benign_01 3

zeek_benign_02: 235 flows -> 11 bundles
{'benign': 11}
skipping tiny capture: zeek_bruteforce_ssh_01 3

zeek_dos_01: 1000 flows -> 50 bundles
{'dos': 50}

zeek_portscan_01: 65554 flows -> 3277 bundles
{'portscan': 3272, 'benign': 5}

zeek_web_01: 90 flows -> 4 bundles
{'benign': 4}


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

FEATS = ["n_flows", "n_dst_ports", "n_dst_hosts", "top_port_share",
         "no_reply_share", "mean_fwd_pkts", "mean_bwd_pkts"]
tr, te = w[~w["is_test"]], w[w["is_test"]]

rfw = RandomForestClassifier(n_estimators=200, min_samples_leaf=2,
                             class_weight="balanced", n_jobs=-1, random_state=42)
rfw.fit(tr[FEATS], tr["family"])
pred = rfw.predict(te[FEATS])
print(classification_report(te["family"], pred, digits=4, zero_division=0))
labels = sorted(w["family"].unique())
print(pd.DataFrame(confusion_matrix(te["family"], pred, labels=labels), index=labels, columns=labels))
print(pd.Series(rfw.feature_importances_, index=FEATS).sort_values(ascending=False).round(4))
joblib.dump({"model": rfw, "features": FEATS}, "../models/rf_window_v0.joblib")

              precision    recall  f1-score   support

      benign     0.9998    0.9776    0.9886     71842
  bruteforce     0.8864    1.0000    0.9398        39
         dos     0.0000    0.0000    0.0000         0
    portscan     0.0000    0.0000    0.0000        15
         web     0.0000    0.0000    0.0000         0

    accuracy                         0.9774     71896
   macro avg     0.3772    0.3955    0.3857     71896
weighted avg     0.9995    0.9774    0.9883     71896

            benign  bruteforce  dos  portscan  web
benign       70231           5    1      1598    7
bruteforce       0          39    0         0    0
dos              0           0    0         0    0
portscan        15           0    0         0    0
web              0           0    0         0    0
n_flows           0.2847
mean_bwd_pkts     0.2286
mean_fwd_pkts     0.1973
no_reply_share    0.1005
top_port_share    0.0845
n_dst_ports       0.0682
n_dst_hosts       0.0362
dtype: float64


['../models/rf_window_v0.joblib']

In [10]:
def zeek_windows(conn):
    d = conn.copy()
    for c in ["orig_pkts", "resp_pkts"]:
        if c not in d.columns: d[c] = np.nan
        d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0)
    d["dst_port"] = pd.to_numeric(d["id.resp_p"], errors="coerce")
    d["minute"] = (pd.to_numeric(d["ts"]) // 60).astype(int)
    d = d.rename(columns={"id.orig_h": "src_ip", "id.resp_h": "dst_ip",
                          "orig_pkts": "fwd", "resp_pkts": "bwd"})
    d["no_reply"] = (d["bwd"] == 0).astype(float)
    return make_windows(d.dropna(subset=["dst_port"]), ["minute", "src_ip"])

LOGS = Path("../data/captures")
for f in sorted(LOGS.rglob("conn.log")):
    conn = pd.read_json(f, lines=True)
    if len(conn) < 50:
        print("skipping tiny capture:", f.parent.name, len(conn)); continue
    ww = zeek_windows(conn)
    p = rfw.predict(ww[FEATS])
    print(f"\n{f.parent.name}: {len(conn)} flows -> {len(ww)} bundles")
    print(pd.Series(p).value_counts().to_dict())

skipping tiny capture: zeek_benign_01 3

zeek_benign_02: 235 flows -> 5 bundles
{'benign': 5}
skipping tiny capture: zeek_bruteforce_ssh_01 3

zeek_dos_01: 1000 flows -> 2 bundles
{'benign': 2}

zeek_portscan_01: 65554 flows -> 7 bundles
{'benign': 7}

zeek_web_01: 90 flows -> 6 bundles
{'benign': 6}
